# 0️⃣5️⃣ - How to manage roles
![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%9B%A0%EF%B8%8F%20Platform%20%2F%20IT%20Administrator-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Intermediate-yellow) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we will explore how to list, create, and assign custom roles, and how to delete roles.

### 🛠️ Before You Begin

If you haven't set up your environment yet, follow the instructions in [SETUP.md](../SETUP.md) before running any cells.

This playbook builds on concepts from:
- [04 — Managing your Admin Users](./04-manage-admins.ipynb)


---

### 📋 What you'll learn

| # | Topic |
|---|-------|
| 1 | How to create a custom role with specific claims |
| 2 | How to update an existing role's properties and claims |
| 3 | How to delete a role |


---

## 🆕 Method 1: Create a Role

**Best for:** Provisioning fine-grained access for admin users who should only have specific capabilities in the RADKit service.

**How it works:**
1. Connect to the service with `ControlAPI.create()`.
2. Import the `Claim` enum to specify the permissions the role should grant.
3. Call `create_role()` with a canonical name, description, and a set of claims.
4. Verify the result by calling `list_roles()`.

**What you need:**
- The RADKit server address and superadmin password
- A canonical role name (lowercase letters, digits, and dashes — cannot start or end with a dash)

> A `claim` defines what a role can access. See the [full list of claims](https://radkit.cisco.com/docs/control_api/control_api.html#radkit_service.control_api.Claim).

The following two roles are enabled by default:

**`basic-admin`**
- Modify Devices, Read Device Templates, Read Activity, Read Roles, Read External Sources, Modify Labels, Read Logs, Read Settings, Modify Remote Users

**`sysadmin`**
- Full access to all system features

### 1.1 Create a new role and list all roles


In [ ]:
from getpass import getpass
from radkit_service.control_api import ControlAPI, Claim

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:

    # Create a new role with only the claim to read devices
    service.create_role(
        name="inventory-wizard",
        description="Device Wizard",
        claims={Claim.READ_DEVICES})

    # Retrieve all the roles available in the service
    for role in service.list_roles().root.result:
        print(f"🍥 Role ({role.id}) name = {role.name}, description = {role.description}, claims = {role.claims}")


🍥 Role (1) name = basic-admin, description = Basic administrator with standard device and user management permissions, claims = frozenset({<Claim.READ_EXTERNAL_SOURCES: 'READ_EXTERNAL_SOURCES'>, <Claim.READ_SETTINGS: 'READ_SETTINGS'>, <Claim.READ_ROLES: 'READ_ROLES'>, <Claim.MODIFY_DEVICES: 'MODIFY_DEVICES'>, <Claim.MODIFY_REMOTE_USERS: 'MODIFY_REMOTE_USERS'>, <Claim.READ_DEVICE_TEMPLATES: 'READ_DEVICE_TEMPLATES'>, <Claim.MODIFY_LABELS: 'MODIFY_LABELS'>, <Claim.READ_ACTIVITY: 'READ_ACTIVITY'>, <Claim.READ_LOGS: 'READ_LOGS'>})
🍥 Role (2) name = sysadmin, description = System administrator with full access to all system features, claims = frozenset()
🍥 Role (3) name = inventory-wizard, description = Device Wizard, claims = frozenset({<Claim.READ_DEVICES: 'READ_DEVICES'>})


---

## ✏️ Method 2: Update a Role

**Best for:** Adjusting an existing role's name, description, or permissions without recreating it from scratch.

**How it works:**
1. Connect to the service with `ControlAPI.create()`.
2. Call `update_role()` with the target `role_id` and the fields to change.
3. Use `add_claims`, `remove_claims`, or `replace_claims` to modify permissions.
4. Verify the result by calling `list_roles()`.

**What you need:**
- The RADKit server address and superadmin password
- The numeric ID of the role to update

### 2.1 Update a role's name, description, and claims


In [ ]:
from getpass import getpass
from radkit_service.control_api import ControlAPI, Claim

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:

    # Update role: rename it and add one more claim
    service.update_role(
        role_id=3,
        name="read-wizard",
        description="Read Only Wizard",
        add_claims={Claim.READ_REMOTE_USERS})

    # Retrieve all the roles available in the service
    for role in service.list_roles().root.result:
        print(f"🍥 Role name = {role.name}, description = {role.description}, claims = {role.claims}")


🍥 Role name = basic-admin, description = Basic administrator with standard device and user management permissions, claims = frozenset({<Claim.READ_EXTERNAL_SOURCES: 'READ_EXTERNAL_SOURCES'>, <Claim.READ_SETTINGS: 'READ_SETTINGS'>, <Claim.READ_ROLES: 'READ_ROLES'>, <Claim.MODIFY_DEVICES: 'MODIFY_DEVICES'>, <Claim.MODIFY_REMOTE_USERS: 'MODIFY_REMOTE_USERS'>, <Claim.READ_DEVICE_TEMPLATES: 'READ_DEVICE_TEMPLATES'>, <Claim.MODIFY_LABELS: 'MODIFY_LABELS'>, <Claim.READ_ACTIVITY: 'READ_ACTIVITY'>, <Claim.READ_LOGS: 'READ_LOGS'>})
🍥 Role name = sysadmin, description = System administrator with full access to all system features, claims = frozenset()
🍥 Role name = read-wizard, description = Read Only Wizard, claims = frozenset({<Claim.READ_REMOTE_USERS: 'READ_REMOTE_USERS'>, <Claim.READ_DEVICES: 'READ_DEVICES'>})


### 2.2 Admin user experience with the new role

Let's create a new admin user and log in to the RADKit server web UI to see the effect of the custom role.


In [ ]:
from getpass import getpass
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:

    service.create_admin(
        enabled=True,
        admin_name="testadmin",
        password="TestAdminPass123",
        email="admin@acme.com",
        full_name="Test Admin",
        description="This is a test admin account created for demonstration purposes.",
        role="read-wizard"
    )


![Roles login](../images/07-roles-login.png)

![Roles denied](../images/08-roles-denied.png)

> As expected, whenever the admin user navigates to a section other than devices or remote users, the content is not available.

---

## 🗑️ Method 3: Delete a Role

**Best for:** Removing unused or outdated roles to keep the role roster clean and manageable.

**How it works:**
1. Connect to the service with `ControlAPI.create()`.
2. Call `delete_role()` with the numeric `role_id` of the role to remove.
3. Verify the result by calling `list_roles()`.

**What you need:**
- The RADKit server address and superadmin password
- The numeric ID of the role to delete

### 3.1 Delete a role and verify removal


In [9]:
from getpass import getpass
from radkit_service.control_api import ControlAPI, Claim

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:

    # Delete the role by its ID
    service.delete_role(role_id=3)

    # Verify the role has been removed
    print("🍥 Remaining roles:")
    for role in service.list_roles().root.result:
        print(f"  🍥 Role ({role.id}) name = {role.name}")


🍥 Remaining roles:
  🍥 Role (1) name = basic-admin
  🍥 Role (2) name = sysadmin


---

## 🔀 Which Method Should I Use?

| Dimension | Method 1: Create | Method 2: Update | Method 3: Delete |
|-----------|-----------------|-----------------|-----------------|
| **Goal** | Define a new role | Modify an existing role | Remove an unused role |
| **Auth required** | Yes (superadmin) | Yes (superadmin) | Yes (superadmin) |
| **Requires role ID** | No | Yes | Yes |
| **Reversible** | Yes (delete later) | Yes (update again) | No |
| **Best for** | Initial setup or new permission scopes | Fine-tuning permissions or renaming | Cleanup of stale or test roles |
